# 仿真数据批量测试 —— PINet 复振幅重建

对 `datasets/data_simulation/` 下的 `.mat` 文件（每个含 (N,256,256) 复振幅数据）进行批量测试：
1. 使用 `diff_compute` 在 z=2mm 处计算衍射图
2. 将衍射图（取模值）输入模型，得到预测复振幅
3. 分别计算振幅和相位相对原始 GT 的 PSNR / SSIM
4. 对每个文件的 N 个 patch 求平均，汇总输出

In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp/PINet_cpx_cl')

## 1. 全局配置

In [ ]:
import os
import numpy as np
from scipy.io import loadmat
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.model import PINet_cpx_v6
from src.config import *
from src.utilities import (
    diff_compute,
    compute_psnr_skimage_single,
    compute_ssim_skimage_single,
)

print(f'Device: {DEVICE}')

In [ ]:
# ==================== 路径配置 ====================
SIM_DATA_DIR = os.path.join(PROJECT_ROOT, 'datasets', 'data_simulation')
CHECKPOINT_PATH = os.path.join(
    SAVE_DIR, 'model_saved_pinet_compared2',
    'model_pinet_size256_epoch_200_batchsize4_1.5mm_to_3mm.pth'
)

# ==================== 物理参数 ====================
Z_MM = 2.0                # 传播距离 (mm)
WAVELENGTH = 532e-9       # 波长 (m)
PIXEL_SIZE = 1.67e-6      # 像素尺寸 (m)
Nx, Ny = 256, 256         # 图像尺寸

# ==================== 模型参数 ====================
FOLD_ITERS = 9            # PINet 物理迭代次数

# ==================== 可视化参数 ====================
PERCENT = 0.5             # 百分位裁剪比例

print(f'数据目录:   {SIM_DATA_DIR}')
print(f'模型路径:   {CHECKPOINT_PATH}')
print(f'传播距离:   {Z_MM} mm')
print(f'波长:       {WAVELENGTH*1e9:.0f} nm')
print(f'像素尺寸:   {PIXEL_SIZE*1e6:.2f} um')
print(f'fold_iters: {FOLD_ITERS}')
print(f'模型文件存在: {os.path.isfile(CHECKPOINT_PATH)}')

## 2. 计算传递函数 (TF) —— z = 2 mm

In [ ]:
z = Z_MM * 1e-3
k = 2 * np.pi / WAVELENGTH
dx = PIXEL_SIZE

Lx = Nx * dx
Ly = Ny * dx
fx = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / Lx, Nx)
fy = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / Ly, Ny)
FX, FY = np.meshgrid(fy, fx)

sqrt_term = k**2 - (2 * np.pi * FX)**2 - (2 * np.pi * FY)**2
sqrt_term = np.maximum(sqrt_term, 0)  # 截断倏逝波分量
TF = np.exp(1j * z * np.sqrt(sqrt_term))
TF_torch = torch.from_numpy(TF).to(DEVICE).to(torch.complex64)

print(f'TF shape: {TF_torch.shape}, dtype: {TF_torch.dtype}')
print(f'z = {z*1e3:.3f} mm, dx = {dx*1e6:.2f} um')
print(f'视场: {Lx*1e3:.2f} mm x {Ly*1e3:.2f} mm')
print(f'Nyquist 频率: {1/(2*dx)*1e-3:.1f} lp/mm')

## 3. 加载模型

In [ ]:
model = PINet_cpx_v6(fold_iters=FOLD_ITERS).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('模型加载完成')

## 4. 批量测试

In [ ]:
def complex_to_torch(arr, device=DEVICE):
    """将 numpy complex 数组转为 torch 复张量 (兼容旧版 PyTorch)。"""
    real = torch.from_numpy(np.real(arr).astype(np.float32))
    imag = torch.from_numpy(np.imag(arr).astype(np.float32))
    return (real + 1j * imag).to(device)


def clip_and_normalize(img, percent=PERCENT):
    """百分位裁剪并归一化到 [0,1]。"""
    vmin = np.percentile(img, percent)
    vmax = np.percentile(img, 100 - percent)
    img = np.clip(img, vmin, vmax)
    return (img - vmin) / (vmax - vmin)


results = []
viz_data = []  # 存储每个文件第一组样本的可视化数据
mat_files = sorted([f for f in os.listdir(SIM_DATA_DIR) if f.endswith('.mat')])
print(f'共找到 {len(mat_files)} 个 .mat 文件\n')

for fname in mat_files:
    file_path = os.path.join(SIM_DATA_DIR, fname)
    data = loadmat(file_path)
    complex_patches = data['complex_patches']  # (N, 256, 256) complex64
    N = complex_patches.shape[0]
    
    amp_psnr_list, amp_ssim_list = [], []
    phs_psnr_list, phs_ssim_list = [], []
    
    for i in range(N):
        gt_complex = complex_patches[i]  # (256, 256) complex64
        gt_tensor = complex_to_torch(gt_complex)
        
        # GT 振幅和相位
        gt_amp = torch.abs(gt_tensor)
        gt_phs = torch.angle(gt_tensor)
        
        # 计算衍射图
        diff = diff_compute(gt_tensor, TF_torch)  # (256, 256) real
        diff = (diff-diff.min())/(diff.max()-diff.min())
        
        # 模型推理
        with torch.no_grad():
            pred, y_rec = model(diff.unsqueeze(0).unsqueeze(0), TF_torch)
        
        pred_np = pred.squeeze().detach().cpu().numpy()
        pred_amp = np.abs(pred_np)
        pred_phs = np.angle(pred_np)
        
        gt_amp_np = gt_amp.detach().cpu().numpy()
        gt_phs_np = gt_phs.detach().cpu().numpy()
        
        # 保存第一组用于可视化
        if i == 0:
            viz_data.append({
                'name': fname.replace('.mat', ''),
                'gt_amp': gt_amp_np,
                'gt_phs': gt_phs_np,
                'pred_amp': pred_amp,
                'pred_phs': pred_phs,
            })
        
        # 计算指标
        amp_psnr_list.append(compute_psnr_skimage_single(torch.from_numpy(pred_amp), gt_amp.cpu()))
        amp_ssim_list.append(compute_ssim_skimage_single(torch.from_numpy(pred_amp), gt_amp.cpu()))
        phs_psnr_list.append(compute_psnr_skimage_single(torch.from_numpy(pred_phs), gt_phs.cpu()))
        phs_ssim_list.append(compute_ssim_skimage_single(torch.from_numpy(pred_phs), gt_phs.cpu()))
    
    results.append({
        '文件': fname,
        'N': N,
        '振幅 PSNR': np.mean(amp_psnr_list),
        '振幅 PSNR_std': np.std(amp_psnr_list),
        '振幅 SSIM': np.mean(amp_ssim_list),
        '振幅 SSIM_std': np.std(amp_ssim_list),
        '相位 PSNR': np.mean(phs_psnr_list),
        '相位 PSNR_std': np.std(phs_psnr_list),
        '相位 SSIM': np.mean(phs_ssim_list),
        '相位 SSIM_std': np.std(phs_ssim_list),
    })
    print(f'{fname}: N={N}, 振幅 PSNR={np.mean(amp_psnr_list):.2f}, SSIM={np.mean(amp_ssim_list):.4f}')

print(f'\n----- 全部 {len(mat_files)} 个文件测试完成 -----')

## 5. 结果展示 —— 标签 vs 预测

In [ ]:
n_files = len(viz_data)
n_cols = 4
fig, axes = plt.subplots(n_files, n_cols, figsize=(16, 4 * n_files))

col_titles = ['Label Amplitude', 'Label Phase', 'Pred Amplitude', 'Pred Phase']

for row_idx, item in enumerate(viz_data):
    gt_amp_disp = clip_and_normalize(item['gt_amp'])
    gt_phs_disp = clip_and_normalize(item['gt_phs'])
    pred_amp_disp = clip_and_normalize(item['pred_amp'])
    pred_phs_disp = clip_and_normalize(item['pred_phs'])
    
    axes[row_idx, 0].imshow(gt_amp_disp, cmap='gray')
    axes[row_idx, 1].imshow(gt_phs_disp, cmap='gray')
    axes[row_idx, 2].imshow(pred_amp_disp, cmap='gray')
    axes[row_idx, 3].imshow(pred_phs_disp, cmap='gray')
    
    # 行标题（左侧标注文件名缩写）
    short_name = item['name'].replace('_complex_patches_256_selected', '')
    axes[row_idx, 0].set_ylabel(short_name, fontsize=8)
    
    for col in range(n_cols):
        axes[row_idx, col].axis('off')

# 列标题
for col in range(n_cols):
    axes[0, col].set_title(col_titles[col], fontsize=12)

plt.tight_layout()
plt.show()

## 6. 汇总结果

In [ ]:
df = pd.DataFrame(results)
df = df.set_index('文件')
df

In [ ]:
# 全局平均
print('===== 全局平均（所有文件）=====')
print(f'振幅 PSNR: {df["振幅 PSNR"].mean():.2f} ± {df["振幅 PSNR"].std():.2f}')
print(f'振幅 SSIM: {df["振幅 SSIM"].mean():.4f} ± {df["振幅 SSIM"].std():.4f}')
print(f'相位 PSNR: {df["相位 PSNR"].mean():.2f} ± {df["相位 PSNR"].std():.2f}')
print(f'相位 SSIM: {df["相位 SSIM"].mean():.4f} ± {df["相位 SSIM"].std():.4f}')